## MISA (2024-2025)
- Alohan'ny mamerina dia avereno atao Run ny notebook iray manontolo. Ny fanaovana azy dia redémarrena mihitsy ny kernel aloha (jereo menubar, safidio **Kernel$\rightarrow$Restart Kernel and Run All Cells**).

- Izay misy hoe `YOUR CODE HERE` na `YOUR ANSWER HERE` ihany no fenoina. Afaka manampy cells vaovao raha ilaina. Aza adino ny mameno references eo ambany raha ilaina.

## References
Eto ilay references rehetra no apetraka
* Nampiasa chagpt

---

In [1]:
from random import randrange
import numpy as np
from sklearn.metrics import mean_squared_error, log_loss
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import normalize


def grad_check_sparse(f, x, analytic_grad, num_checks=10, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [2]:
data = load_diabetes()
X, y = data.data, data.target

In [8]:
def mse_loss_vectorized(w, b, X, y):
    """
    MSE loss function WITHOUT FOR LOOPs , NO REGULARIZATION
    
    Returns a tuple of:
    - loss 
    - gradient with respect to weights w
    - gradient with respect to bias b
    """
    loss = 0.0
    dw = np.zeros_like(w)
    
    # YOUR CODE HERE
    mse = np.mean((y - X@w - b)**2)
    loss = mse
    dw = -2 * X.T @ (y - X @ w - b) / len(y)
    db = -2 * np.mean(y - X @ w -b) 
    
    return loss, dw, np.array(db).reshape(1,)

In [9]:
def soft_threshold(x, delta):
    # YOUR CODE HERE
    return np.sign(x) * np.maximum(np.abs(x) - delta, 0)

# Lasso Subgradient Descent

In [11]:
def l1_subgradient(w):
    """
    Subgradient of the L1 loss
    """
    dw = np.zeros_like(w)
    # YOUR CODE HERE
    
    return np.sign(w)
    

def lasso_subgradient_mse_loss_vectorized(w, b, X, y, alpha):
    """
    MSE loss function adding the subgradient for w
    """
    loss, dw, db = mse_loss_vectorized(w, b, X, y)
    # Add the subgradient to dw
    # YOUR CODE HERE
    dw += alpha * l1_subgradient(w) 
    return loss, dw, db

In [30]:
class LassolSubgradientDescent():
    def __init__(self,  alpha=0.1):
        self.w = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            # YOUR CODE HERE
            idx = np.random.choice(N, batch_size, replace=False)
            X_batch = X[idx]
            y_batch = y[idx]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # YOUR CODE HERE
            self.w -= learning_rate * dw
            self.b -= learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
    
    
    def predict(self, X):
        # YOUR CODE HERE
        return X @ self.w + self.b

    def loss(self, X_batch, y_batch):
        return lasso_subgradient_mse_loss_vectorized(self.w, self.b, X_batch, y_batch, self.alpha)

In [31]:
model = LassolSubgradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse = mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 28652.809296
iteration 10000 / 200000: loss 3501.735451
iteration 20000 / 200000: loss 2964.002526
iteration 30000 / 200000: loss 3065.418646
iteration 40000 / 200000: loss 3044.588161
iteration 50000 / 200000: loss 2930.411497
iteration 60000 / 200000: loss 2947.969244
iteration 70000 / 200000: loss 2918.126038
iteration 80000 / 200000: loss 2779.852214
iteration 90000 / 200000: loss 2783.237611
iteration 100000 / 200000: loss 2944.909154
iteration 110000 / 200000: loss 2760.550296
iteration 120000 / 200000: loss 3158.910372
iteration 130000 / 200000: loss 2995.506279
iteration 140000 / 200000: loss 2770.303185
iteration 150000 / 200000: loss 2887.137770
iteration 160000 / 200000: loss 3394.787142
iteration 170000 / 200000: loss 3013.187543
iteration 180000 / 200000: loss 2818.278604
iteration 190000 / 200000: loss 2922.173305
MSE scikit-learn: 2912.5274926036195
MSE Coordinate descent model : 2888.7155084707697


In [32]:
model = LassolSubgradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse = mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 28218.382982
iteration 10000 / 200000: loss 4447.139073
iteration 20000 / 200000: loss 4262.301293
iteration 30000 / 200000: loss 3920.909545
iteration 40000 / 200000: loss 3869.435759
iteration 50000 / 200000: loss 3630.907040
iteration 60000 / 200000: loss 4259.706869
iteration 70000 / 200000: loss 3988.744651
iteration 80000 / 200000: loss 4080.954675
iteration 90000 / 200000: loss 3750.962470
iteration 100000 / 200000: loss 3555.912658
iteration 110000 / 200000: loss 3975.996124
iteration 120000 / 200000: loss 3650.405894
iteration 130000 / 200000: loss 4088.712675
iteration 140000 / 200000: loss 3661.363220
iteration 150000 / 200000: loss 3363.117885
iteration 160000 / 200000: loss 3795.996237
iteration 170000 / 200000: loss 3820.937851
iteration 180000 / 200000: loss 3692.823730
iteration 190000 / 200000: loss 3627.563075
MSE scikit-learn: 5650.290772564549
MSE Coordinate descent model : 3810.95951234814


# Lasso Proximal Gradient Descent (iterative soft thresholding algorithm or ISTA)

In [33]:
class LassoProximalGradientDescent():
    def __init__(self,  alpha=0.1):
        self.w = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            # YOUR CODE HERE
            idx = np.random.choice(N, batch_size, replace=False)
            X_batch = X[idx]
            y_batch = y[idx]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # PROJECT THE GRADIENT FOR w USING soft_threshold
            # YOUR CODE HERE
            delta = learning_rate * self.alpha
            w_t = self.w - learning_rate * dw
            self.w = soft_threshold(w_t, delta)
            self.b -= learning_rate * db
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))

    def predict(self, X):
        # YOUR CODE HERE
        return X @ self.w + self.b

    def loss(self, X_batch, y_batch):
        return mse_loss_vectorized(self.w, self.b, X_batch, y_batch)

In [34]:
model = LassoProximalGradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 29878.368632
iteration 10000 / 200000: loss 3290.148753
iteration 20000 / 200000: loss 3328.572654
iteration 30000 / 200000: loss 3251.031820
iteration 40000 / 200000: loss 2871.703901
iteration 50000 / 200000: loss 2812.648739
iteration 60000 / 200000: loss 2709.576636
iteration 70000 / 200000: loss 2997.853900
iteration 80000 / 200000: loss 3234.200422
iteration 90000 / 200000: loss 2793.438808
iteration 100000 / 200000: loss 2848.853496
iteration 110000 / 200000: loss 2743.227931
iteration 120000 / 200000: loss 3044.320926
iteration 130000 / 200000: loss 2610.224868
iteration 140000 / 200000: loss 3075.169367
iteration 150000 / 200000: loss 2738.862623
iteration 160000 / 200000: loss 2981.703954
iteration 170000 / 200000: loss 2907.213750
iteration 180000 / 200000: loss 2742.441319
iteration 190000 / 200000: loss 3074.464188
MSE scikit-learn: 2912.5274926036195
MSE Coordinate descent model : 2888.6470224197546


In [35]:
model = LassoProximalGradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 28182.884524
iteration 10000 / 200000: loss 4735.503874
iteration 20000 / 200000: loss 4122.629348
iteration 30000 / 200000: loss 3791.951578
iteration 40000 / 200000: loss 4321.231564
iteration 50000 / 200000: loss 3742.956928
iteration 60000 / 200000: loss 3706.337827
iteration 70000 / 200000: loss 3684.149355
iteration 80000 / 200000: loss 3554.972892
iteration 90000 / 200000: loss 3474.798335
iteration 100000 / 200000: loss 3655.068419
iteration 110000 / 200000: loss 3665.671514
iteration 120000 / 200000: loss 3860.967366
iteration 130000 / 200000: loss 3813.684796
iteration 140000 / 200000: loss 3761.847785
iteration 150000 / 200000: loss 3812.367594
iteration 160000 / 200000: loss 3917.448270
iteration 170000 / 200000: loss 4088.092476
iteration 180000 / 200000: loss 3801.706519
iteration 190000 / 200000: loss 3197.540618
MSE scikit-learn: 5650.290772564549
MSE Coordinate descent model : 3811.1761610115736


# Lasso Projected Gradient Descent

In [38]:
class LassoProjectedGradientDescent():
    def __init__(self,  alpha=0.1):
        self.w_p = None
        self.w_n = None
        self.b = None
        self.alpha = alpha

    def train(self, X, y, learning_rate=1e-3, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w_p is None: # Initialization
            self.w_p = 0.001 * np.random.randn(d)
            self.w_n = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            # YOUR CODE HERE
            idx = np.random.choice(N, batch_size, replace=True)
            X_batch = X[idx]
            y_batch = y[idx]
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.  
            # Project for w_p and w_n
            # YOUR CODE HERE
            # Gradient wrt w_p and w_n
            dw_p = dw
            dw_n = -dw

            # Gradient step 
            self.w_p -= learning_rate * dw_p
            self.w_n -= learning_rate * dw_n
            self.b   -= learning_rate * db

            # Projection 
            self.w_p = np.maximum(self.w_p, 0)
            self.w_n = np.maximum(self.w_n, 0)
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))

    def predict(self, X):
        # YOUR CODE HERE
        w = self.w_p - self.w_n
        return X @ w + self.b

    def loss(self, X_batch, y_batch):
        w = self.w_p - self.w_n
        n = len(y_batch)

        # Prediction
        y_pred = X_batch @ w + self.b
        error = y_batch - y_pred

        # Loss
        mse = np.mean(error**2)
        l1 = self.alpha * np.sum(self.w_p + self.w_n)
        loss = mse + l1

        # Gradient
        dw = -2/n * X_batch.T @ error
        db = -2/n * np.sum(error)
        return loss, dw, db
        

In [39]:
model = LassoProjectedGradientDescent(alpha=0.1)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

iteration 0 / 200000: loss 29977.545373
iteration 10000 / 200000: loss 3737.208690
iteration 20000 / 200000: loss 3011.126969
iteration 30000 / 200000: loss 3184.983099
iteration 40000 / 200000: loss 2985.641169
iteration 50000 / 200000: loss 2756.339648
iteration 60000 / 200000: loss 3106.986044
iteration 70000 / 200000: loss 3225.101398
iteration 80000 / 200000: loss 2753.550069
iteration 90000 / 200000: loss 3120.761474
iteration 100000 / 200000: loss 3000.928015
iteration 110000 / 200000: loss 2770.883567
iteration 120000 / 200000: loss 3260.962165
iteration 130000 / 200000: loss 2893.985844
iteration 140000 / 200000: loss 3350.427604
iteration 150000 / 200000: loss 3137.763081
iteration 160000 / 200000: loss 3484.023929
iteration 170000 / 200000: loss 3127.261703
iteration 180000 / 200000: loss 3488.757290
iteration 190000 / 200000: loss 2875.761322
MSE scikit-learn: 2912.5274926036195
MSE Coordinate descent model : 2878.795198438797


In [ ]:
model = LassoProjectedGradientDescent(alpha=2)
model.train(X, y, learning_rate=1e-2,verbose=True, num_iters=200_000)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=True)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

# Lasso Coordinate Descent (no intercept)

In [46]:
class LassoCoordinateDescent():
    def __init__(self, alpha=0.1):
        self.w = None
        self.alpha = alpha

    def train(self, X, y, num_iters=1000):
        N, d = X.shape
        
        # YOUR CODE HERE
        # Initialize weights
        if self.w is None:
            self.w = np.zeros(d)

        # Precompute X_j^T X_j for efficiency
        X_norm_square = np.sum(X**2, axis=0)

        for it in range(num_iters):
            for j in range(d):
                # Compute residual without contribution of w_j
                r_j = y - (X @ self.w - X[:, j] * self.w[j])

                # Compute w_j^OLS
                w_j_OLS = np.dot(X[:, j], r_j) / X_norm_square[j]

                # Soft-threshold
                delta = self.alpha / X_norm_square[j]
                self.w[j] = soft_threshold(w_j_OLS, delta)

    def predict(self, X): 
        # YOUR CODE HERE
        return X @ self.w

In [44]:
model = LassoCoordinateDescent(alpha=0.1)
model.train(X, y)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=0.1, fit_intercept=False)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

MSE scikit-learn: 26057.124496145752
MSE Coordinate descent model : 26004.303657948607


In [45]:
model = LassoCoordinateDescent(alpha=2)
model.train(X, y)
pred = model.predict(X)
mse= mean_squared_error(pred, y)

sk_model = Lasso(alpha=2, fit_intercept=False)
sk_model.fit(X, y)
sk_pred = sk_model.predict(X)
sk_mse = mean_squared_error(sk_pred, y)

print("MSE scikit-learn:", sk_mse)
print("MSE Coordinate descent model :", mse)
assert mse - sk_mse < 50

MSE scikit-learn: 28794.887776106683
MSE Coordinate descent model : 26006.422489676806
